# Ensamble analysis V1 pasive imaging 𓆝 𓆟 𓆞 𓆝 𓆟𓆝 𓆟 𓆞 𓆝 𓆟
---
This notebook shows the analysis pipeline for searching ensambles on two-photon calcium imaging data. This analysis is based on the results of analyses of individual neurons and pairs of neurons. These analyses revealed that BTBR exhibited reduced selectivity for orientation and direction, as well as greater noise correlation than the control group, raising questions regarding the level of population organization. This notebook assumes 2 experimental groups. 
The main question that guides this analysis is Can both the BTBR mice and C57B6 mice show ensamblatic activity on a pasive stimulation task? *Is equaly organized represented in both genotypes?*. 

## Imports and general setup

We first need to import the Python libraries we will use in the rest of the notebook.

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt   
from scipy import stats
from sklearn.covariance import LedoitWolf
from pathlib import Path

In [9]:
# FUNTION MODULE
def find_file( folder: Path, keyword: str, preferred_extension: str = None) -> Path:
    """
    Searches within `folder` for a single file whose name contains `keyword`
    (case-insensitive).

    If there are multiple candidates and `preferred_extension` is specified, the results are filtered
    by that extension before a decision is made. If, after filtering, there are still
    more than one, or if `preferred_extension` is not specified and there is more than one candidate,
    an error is raised: the ambiguity must be resolved manually; the system never simply selects
    “the first one that appears.”
    """
    kw = keyword.lower()
    matches= [f for f in folder.iterdir() if f.is_file() and kw in f.name.lower()]

    if not matches:
        raise FileNotFoundError(f"Could'nt find any file with {keyword} in {folder}.")

    if preferred_extension and len(matches) > 1:
        filtered = [f for f in matches if f.suffix.lower() == preferred_extension.lower()]
        if filtered:
            matches = filtered

    if len(matches) > 1:
        names = [f.name for f in matches]
        raise ValueError(
            f"Ambiguity when searching for ‘{keyword}’ in {folder}: {len(matches)} candidates {names}. "
            "Resolve this manually before continuing"
        )
    return matches[0]

def detect_genotype(group_name: str) -> list:
    for kw, geno in GENOTYPE_FOLDERS.items():
        if kw.upper() in group_name.upper():
             raise ValueError(
                f"the genotype '{group_name}' in the folder can't be recognized."
                f"Known folders: {list(GENOTYPE_FOLDES.keys())}"
            )

def fdiscover_animals (root: Path, genotype: str) -> list:
    """
    Discover all animals in the root folder for a given genotype.
    """
    root= Path (root)
    animals= []
    for group in sorted(root.iterdir()):
        if not group.is_dir() or group.name.startswith(".") or group.name== "pipeline_output":
            continue
        genotype = detect_genotype(group.name)
        for animal in sorted(group.iterdir()):
            if not animal.is_dir() or animal.name.startswith("."):
                continue
            animals.append({
                'Id': animal.name,
                'genotype': genotype,
                'group': group.name,
                'folder': animal,
            })
    if not animals:
        raise ValueError(f"No animals found in {root} for genotype {genotype}.")
    return animals